# Customer Churn Prediction — End-to-End Analysis & Modeling

This notebook demonstrates the end-to-end machine learning lifecycle for customer churn prediction using the **Telco Customer Churn** dataset.

### Pipeline Outline:
1. Environment Setup & Data Loading
2. Exploratory Data Analysis & Schema Verification
3. Leakage-Free Preprocessing (`ColumnTransformer`, `StandardScaler`, `OneHotEncoder`)
4. Model Training & Comparison (`RandomForest`, `DecisionTree`, `XGBoost`)
5. Evaluation & Performance Metrics (ROC-AUC, F1-Score, Confusion Matrix)
6. Model Serialization & Export

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
data_path = "../data/raw/churn.csv"
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset Shape: 7044 rows, 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
churn_counts = df["Churn"].value_counts(normalize=True) * 100
print("Churn Distribution (%):")
print(churn_counts)

Churn Distribution (%):
Churn
No     73.46678
Yes    26.53322
Name: proportion, dtype: float64


In [4]:
clean_df = df.copy()
if "customerID" in clean_df.columns:
    clean_df = clean_df.drop(columns=["customerID"])

clean_df["TotalCharges"] = pd.to_numeric(clean_df["TotalCharges"], errors="coerce").fillna(0)
clean_df["Churn"] = clean_df["Churn"].apply(lambda x: 1 if str(x).strip().lower() == "yes" else 0).astype(int)

X = clean_df.drop(columns=["Churn"])
y = clean_df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

Train Shape: (5635, 19), Test Shape: (1409, 19)


In [5]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=0)), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("ohe", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))]), cat_cols),
    ]
)

X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)
print(f"Transformed Training Feature Shape: {X_train_final.shape}")

Transformed Training Feature Shape: (5635, 30)


In [6]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric="logloss")
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_final, y_train)
    trained_models[name] = model
    y_pred = model.predict(X_test_final)
    y_prob = model.predict_proba(X_test_final)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
comparison_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Random Forest,0.755855,0.527574,0.767380,0.625272,0.843349
1,Decision Tree,0.716111,0.479167,0.799465,0.599198,0.831344
2,XGBoost,0.807665,0.678201,0.524064,0.591252,0.847458


In [7]:
best_rf = trained_models["Random Forest"]
y_pred_best = best_rf.predict(X_test_final)
cm = confusion_matrix(y_test, y_pred_best)

print("Random Forest Confusion Matrix:")
print(cm)

Random Forest Confusion Matrix:
[[778 257]
 [ 87 287]]


In [8]:
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_rf)
])

os.makedirs("../models", exist_ok=True)
joblib.dump(full_pipeline, "../models/churn_full_pipeline.pkl")
print("Full deployable pipeline saved successfully to models/churn_full_pipeline.pkl")

Full deployable pipeline saved successfully to models/churn_full_pipeline.pkl
